# Compare Ensino Fundamental vs Ensino Médio

**Living Stone Foundation — Applied Data Lab**

**Audience:** team leads / reviewers who need a clear reason for **two models and two Streamlit apps**.

**Core question:** Do abandonment levels and school-trait associations differ enough between Fundamental and Médio that a single pooled Brazil model would be misleading?


## Variable dictionary (read before the charts)

| Column / label in charts | Plain-English meaning | How to read values |
|---|---|---|
| `target_dropout_rate` / **Abandonment rate (%)** | Official INEP *taxa de abandono*: share of students who **stopped attending** during the school year after the census reference date | 0 = nobody left; 5 = 5% left. Higher = worse |
| `year` | School census / rendimento year | Years present in the mart (currently 2018–2025) |
| `school_id` (`CO_ENTIDADE`) | Unique school code (INEP) | Join key between Censo and Rendimento |
| `uf` | Brazilian state abbreviation | e.g. SP, BA, AM |
| `municipio_id` | IBGE municipality code | Geographic context |
| `tp_dependencia` | Administrative network | 1=Federal, 2=State, 3=Municipal, 4=Private |
| `tp_localizacao` | School location type | 1=Urban, 2=Rural |
| `is_rural` | 1 if rural school | Shortcut of `tp_localizacao == 2` |
| `is_public` | 1 if public network (federal/state/municipal) | 0 = private |
| `in_agua` | Has potable / public water | 1=yes, 0=no |
| `in_energia` | Connected to public electricity | 1=yes, 0=no |
| `in_esgoto` | Public sewage connection | 1=yes, 0=no |
| `in_internet` | Internet available at school | 1=yes, 0=no |
| `in_biblioteca` | Library / reading room | 1=yes, 0=no |
| `in_lab_info` | Computer lab | 1=yes, 0=no |
| `in_quadra` | Sports court | 1=yes, 0=no |
| `qt_mat_bas` | Enrollment count (basic education total, when available) | Larger = bigger school |
| `enrollment_level` | Enrollment used for this level (Fundamental or Médio) | Filter requires ≥ 20 students |
| `qt_doc_bas` | Number of teachers (basic education) | Staffing intensity |
| `student_teacher_ratio` | Students ÷ teachers | Higher often means more crowded classes |
| `risk_band` | low / moderate / high | Relative triage label inside the level |
| `high_risk` | 1 if school is in the elevated-risk group | Binary flag for triage demos |
| **MAE / RMSE / R²** | Model error metrics (later notebooks / model folder) | Lower MAE/RMSE better; R² closer to 1 better |

### Acronyms
| Acronym | Meaning |
|---|---|
| **INEP** | Brazilian federal education statistics institute |
| **Censo Escolar** | Annual school census (structure + enrollment) |
| **Taxas de Rendimento** | Official approval / failure / abandonment rates |
| **EDA** | Exploratory Data Analysis |
| **UF** | Federative unit (state) |
| **Fundamental** | Ensino Fundamental (approx. primary + lower secondary) |
| **Médio** | Ensino Médio (upper secondary) |


In [ ]:
# Cell A — project path only (no Path.cwd / exists / resolve)
import sys
ROOT = r"C:\Users\User\Desktop\Projeto Living Stone Foundation"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


### What this cell did
Added the project folder to Python’s import path using a **fixed string** (no `Path.cwd()` / `exists` / `resolve`). Those path checks can freeze kernels on Desktop/OneDrive.


In [ ]:
# Cell B — imports (first run can take a minute for pandas/seaborn)
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

FEATURE_LABELS = {
    "is_rural": "Rural school (1=yes)",
    "is_public": "Public network (1=yes)",
    "in_internet": "Has internet (1=yes)",
    "in_lab_info": "Has computer lab (1=yes)",
    "in_quadra": "Has sports court (1=yes)",
    "in_biblioteca": "Has library (1=yes)",
    "in_agua": "Has water (1=yes)",
    "in_energia": "Has electricity (1=yes)",
    "in_esgoto": "Has sewage (1=yes)",
    "enrollment_level": "Enrollment (this level)",
    "qt_mat_bas": "Basic-ed enrollment (total)",
    "qt_doc_bas": "Number of teachers",
    "student_teacher_ratio": "Students per teacher",
    "tp_dependencia": "Admin network code",
    "tp_localizacao": "Urban/rural code",
    "target_dropout_rate": "Abandonment rate (%)",
}
from src.eda import compare_levels, load_level_mart
print("Imports OK")


### What this cell did
Loaded charting/table libraries. The **first** run can take ~30–90s while pandas/seaborn warm up — that is normal, not a freeze on `import sys`.


In [ ]:
summary = compare_levels().rename(columns={
    'level': 'Education level',
    'rows': 'School-year rows',
    'schools': 'Unique schools',
    'mean_abandono': 'Mean abandonment (%)',
    'median_abandono': 'Median abandonment (%)',
    'p90_abandono': '90th percentile (%)',
    'public_share': 'Share public',
    'rural_share': 'Share rural',
})
display(summary)
fund = load_level_mart('fundamental')
med = load_level_mart('medio')
print('Loaded Fundamental rows:', len(fund), '| Medio rows:', len(med))


### How to read this summary
Compare the two rows:
- **Mean / 90th percentile abandonment** — is Médio systematically “harder”?
- **Row counts** — Médio has fewer schools (only those offering upper secondary).
- **Public / rural shares** — different school mixes can change which features matter.

### Insight
A higher Médio mean with fewer schools already suggests **different operating regimes**. That is necessary (but not sufficient) evidence for separate models — the next charts check the shape of risk and the correlates.


## 1. Outcome shape: boxplots side by side


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.boxplot(data=fund, y='target_dropout_rate', ax=ax[0], color='#4c78a8')
ax[0].set_title('Fundamental — abandonment rate (%)')
ax[0].set_ylabel('Official abandonment rate (%)')
sns.boxplot(data=med, y='target_dropout_rate', ax=ax[1], color='#e76f51')
ax[1].set_title('Médio — abandonment rate (%)')
ax[1].set_ylabel('')
plt.suptitle('Are the risk distributions similar?', y=1.02)
plt.tight_layout()
plt.show()
print('Mean Fundamental: {:.2f}%'.format(fund['target_dropout_rate'].mean()))
print('Mean Médio:       {:.2f}%'.format(med['target_dropout_rate'].mean()))
print('P90 Fundamental:  {:.2f}%'.format(fund['target_dropout_rate'].quantile(0.9)))
print('P90 Médio:        {:.2f}%'.format(med['target_dropout_rate'].quantile(0.9)))


### How to read a boxplot (30-second guide)
- The **box** = middle 50% of schools.
- The line inside = **median** (half the schools are below it).
- Dots / whiskers show the spread and extreme schools.

If Médio’s box and whiskers sit **higher** or stretch farther than Fundamental’s, the levels do not share the same typical risk.

### Insight
When the distributions differ, a pooled model tends to under-serve one level (often Médio). Separate models keep errors and explanations honest per stage.


## 2. Do the associated school traits differ?


In [ ]:
def top_corr(df):
    cols = ['is_rural','is_public','in_internet','in_lab_info','in_quadra','in_biblioteca','enrollment_level']
    tmp = df.copy()
    tmp['student_teacher_ratio'] = tmp['enrollment_level'] / tmp['qt_doc_bas'].replace({0: pd.NA})
    cols = [c for c in cols + ['student_teacher_ratio'] if c in tmp.columns]
    s = tmp[cols + ['target_dropout_rate']].corr(numeric_only=True)['target_dropout_rate'].drop('target_dropout_rate')
    return s.rename(index=FEATURE_LABELS)

cmp = pd.DataFrame({
    'Fundamental': top_corr(fund),
    'Médio': top_corr(med),
})
display(cmp)
ax = cmp.plot(kind='barh', figsize=(10, 6))
ax.set_title('Same school traits, two education levels — correlation with abandonment')
ax.set_xlabel('Correlation with abandonment rate (−1 to +1)')
plt.tight_layout()
plt.show()


### How to read this comparison chart
- Each horizontal pair of bars is **one school trait** (see dictionary above).
- Blue-ish / left series = **Fundamental**; orange / other series = **Médio** (legend in the plot).
- Ask:
  1. Do the **signs** agree (both positive or both negative)?
  2. Do the **magnitudes** differ a lot?
  3. Is the **ranking** of strongest traits different?

### Insight (argument for two Streamlit apps)
If Fundamental is more tied to **infrastructure / rurality** while Médio is more tied to **school size / staffing (students per teacher)**, then:
- counselors need **different explanation templates** per level;
- one shared “top 5 drivers” list would mislead users;
- therefore **two models + two apps** is the scientifically cleaner product design.

Still remember: correlations are associations, not proof of cause.


## 3. Trained-model drivers (if models already exist)


In [ ]:
rows = []
for level, label in [('fundamental','Fundamental'), ('medio','Médio')]:
    path = Path(ROOT) / 'models' / level / 'figures' / 'global_importance_top.csv'
    try:
        imp = pd.read_csv(path).head(8)
        imp['level'] = label
        imp['feature_label'] = imp['feature'].map(FEATURE_LABELS).fillna(imp['feature'])
        rows.append(imp)
    except FileNotFoundError:
        pass
if rows:
    both = pd.concat(rows, ignore_index=True)
    display(both[['level','feature_label','importance']])
else:
    print('Train models first: python -m src.train --level both')


### How to read this table
For each level, features are ordered by how much the **selected model** relies on them when predicting abandonment.

### Final insight for reviewers
1. Official labels + Brazil basic education ⇒ population/data alignment restored.  
2. Fundamental and Médio differ in **level** and often in **drivers**.  
3. Dual products are a narrowing of scope (one country), not a return to multi-country sprawl.  
4. Apps must show the limitation: **school triage**, not student scoring.
